In [9]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import joblib

SEED = 42
np.random.seed(SEED)

# Загрузка
X_train = pd.read_csv('../data/processed/X_train.csv')
X_val = pd.read_csv('../data/processed/X_val.csv')
X_test = pd.read_csv('../data/processed/X_test.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()
y_val = pd.read_csv('../data/processed/y_val.csv').squeeze()
y_test = pd.read_csv('../data/processed/y_test.csv').squeeze()

# Определяем типы колонок
numeric_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"Числовых признаков: {len(numeric_cols)}")
print(f"Категориальных: {len(categorical_cols)}")
if len(categorical_cols) > 0:
    print("Примеры:", categorical_cols[:5])

# Препроцессор (импутация + масштабирование для чисел, OHE для категорий)
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_cols),
    ('cat', categorical_transformer, categorical_cols)
])

def get_pipeline(model):
    return Pipeline([('prep', preprocessor), ('reg', model)])

# Модели с увеличенной сложностью
models = {
    'Ridge': Ridge(alpha=1.0, random_state=SEED),
    'ElasticNet': ElasticNet(alpha=0.01, l1_ratio=0.5, random_state=SEED, max_iter=5000),
    'RandomForest': RandomForestRegressor(n_estimators=150, max_depth=15, random_state=SEED, n_jobs=-1),
    'GradientBoosting': GradientBoostingRegressor(n_estimators=150, max_depth=5, learning_rate=0.05, random_state=SEED)
}

results = {}
best_model = None
best_r2 = -np.inf

for name, model in models.items():
    pipe = get_pipeline(model)
    pipe.fit(X_train, y_train)
    y_pred_val = pipe.predict(X_val)
    r2 = r2_score(y_val, y_pred_val)
    results[name] = r2
    print(f"{name:15} | R² val = {r2:.4f}")
    if r2 > best_r2:
        best_r2 = r2
        best_model = pipe
    joblib.dump(pipe, f'../models/{name.lower()}_model.pkl')

# Оценка на тесте
y_pred_test = best_model.predict(X_test)
r2_test = r2_score(y_test, y_pred_test)
mae_test = mean_absolute_error(y_test, y_pred_test)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
print(f"\nЛучшая модель на ТЕСТЕ: R² = {r2_test:.4f}, MAE = {mae_test:.2f}, RMSE = {rmse_test:.2f}")

/var/folders/t9/w50p8w0917b0t61ks_6y_3g40000gn/T/ipykernel_24907/2234289987.py:25: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()


Числовых признаков: 9
Категориальных: 7
Примеры: ['order_channel', 'store_location_type', 'region', 'customer_age_group', 'customer_gender']
Ridge           | R² val = 0.9501
ElasticNet      | R² val = 0.9500
RandomForest    | R² val = 0.9512
GradientBoosting | R² val = 0.9533

Лучшая модель на ТЕСТЕ: R² = 0.9550, MAE = 0.97, RMSE = 1.18
